<a href="https://colab.research.google.com/github/DaviRoberto10/Atividade-de-Algoritmo-e-Complexidade/blob/Sistema-de-Navega%C3%A7%C3%A3o-de-Rotas-e-Dados-Hier%C3%A1rquicos/Sistema_de_Navega%C3%A7%C3%A3o_de_Rotas_e_Dados_Hier%C3%A1rquicos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# ---------- Árvore AVL ----------
class NoAVL:
    def __init__(self, chave, valor=None):
        self.chave = chave
        self.valor = valor
        self.esq = None
        self.dir = None
        self.altura = 1

class ArvoreAVL:
    def __init__(self):
        self.raiz = None

    def altura(self, no):
        return no.altura if no else 0

    def fator_balanceamento(self, no):
        return self.altura(no.esq) - self.altura(no.dir) if no else 0

    def rotacao_direita(self, y):
        x = y.esq
        T2 = x.dir
        x.dir = y
        y.esq = T2
        y.altura = 1 + max(self.altura(y.esq), self.altura(y.dir))
        x.altura = 1 + max(self.altura(x.esq), self.altura(x.dir))
        return x

    def rotacao_esquerda(self, x):
        y = x.dir
        T2 = y.esq
        y.esq = x
        x.dir = T2
        x.altura = 1 + max(self.altura(x.esq), self.altura(x.dir))
        y.altura = 1 + max(self.altura(y.esq), self.altura(y.dir))
        return y

    def inserir(self, raiz, chave, valor=None):
        if not raiz:
            return NoAVL(chave, valor)
        if chave < raiz.chave:
            raiz.esq = self.inserir(raiz.esq, chave, valor)
        elif chave > raiz.chave:
            raiz.dir = self.inserir(raiz.dir, chave, valor)
        else:
            raiz.valor = valor
            return raiz

        raiz.altura = 1 + max(self.altura(raiz.esq), self.altura(raiz.dir))
        fb = self.fator_balanceamento(raiz)

        # Rotações
        if fb > 1 and chave < raiz.esq.chave:
            return self.rotacao_direita(raiz)
        if fb < -1 and chave > raiz.dir.chave:
            return self.rotacao_esquerda(raiz)
        if fb > 1 and chave > raiz.esq.chave:
            raiz.esq = self.rotacao_esquerda(raiz.esq)
            return self.rotacao_direita(raiz)
        if fb < -1 and chave < raiz.dir.chave:
            raiz.dir = self.rotacao_direita(raiz.dir)
            return self.rotacao_esquerda(raiz)

        return raiz

    def remover(self, raiz, chave):
        if not raiz:
            return raiz
        if chave < raiz.chave:
            raiz.esq = self.remover(raiz.esq, chave)
        elif chave > raiz.chave:
            raiz.dir = self.remover(raiz.dir, chave)
        else:
            if not raiz.esq:
                return raiz.dir
            elif not raiz.dir:
                return raiz.esq
            temp = self.minimo(raiz.dir)
            raiz.chave, raiz.valor = temp.chave, temp.valor
            raiz.dir = self.remover(raiz.dir, temp.chave)
        if not raiz:
            return raiz

        raiz.altura = 1 + max(self.altura(raiz.esq), self.altura(raiz.dir))
        fb = self.fator_balanceamento(raiz)

        if fb > 1 and self.fator_balanceamento(raiz.esq) >= 0:
            return self.rotacao_direita(raiz)
        if fb > 1 and self.fator_balanceamento(raiz.esq) < 0:
            raiz.esq = self.rotacao_esquerda(raiz.esq)
            return self.rotacao_direita(raiz)
        if fb < -1 and self.fator_balanceamento(raiz.dir) <= 0:
            return self.rotacao_esquerda(raiz)
        if fb < -1 and self.fator_balanceamento(raiz.dir) > 0:
            raiz.dir = self.rotacao_direita(raiz.dir)
            return self.rotacao_esquerda(raiz)
        return raiz

    def minimo(self, no):
        while no.esq:
            no = no.esq
        return no

    def busca(self, raiz, chave):
        if not raiz:
            return None
        if chave == raiz.chave:
            return raiz
        elif chave < raiz.chave:
            return self.busca(raiz.esq, chave)
        else:
            return self.busca(raiz.dir, chave)

    def inorder(self, no):
        if not no: return []
        return self.inorder(no.esq) + [(no.chave, no.valor)] + self.inorder(no.dir)

    def preorder(self, no):
        if not no: return []
        return [(no.chave, no.valor)] + self.preorder(no.esq) + self.preorder(no.dir)

    def postorder(self, no):
        if not no: return []
        return self.postorder(no.esq) + self.postorder(no.dir) + [(no.chave, no.valor)]


# ---------- Grafo ----------
import heapq

class Grafo:
    def __init__(self):
        self.adj = {}

    def add_vertice(self, v):
        if v not in self.adj:
            self.adj[v] = []

    def add_aresta(self, u, v, peso=1):
        self.add_vertice(u)
        self.add_vertice(v)
        self.adj[u].append((v, peso))
        self.adj[v].append((u, peso))

    def bfs(self, inicio):
        visitados = []
        fila = [inicio]
        while fila:
            atual = fila.pop(0)
            if atual not in visitados:
                visitados.append(atual)
                for vizinho, _ in self.adj[atual]:
                    if vizinho not in visitados:
                        fila.append(vizinho)
        return visitados

    def dfs(self, inicio):
        visitados = []
        pilha = [inicio]
        while pilha:
            atual = pilha.pop()
            if atual not in visitados:
                visitados.append(atual)
                for vizinho, _ in self.adj[atual]:
                    pilha.append(vizinho)
        return visitados

    def dijkstra(self, inicio):
        dist = {v: float('inf') for v in self.adj}
        dist[inicio] = 0
        pq = [(0, inicio)]
        while pq:
            d, u = heapq.heappop(pq)
            if d > dist[u]: continue
            for v, peso in self.adj[u]:
                nd = d + peso
                if nd < dist[v]:
                    dist[v] = nd
                    heapq.heappush(pq, (nd, v))
        return dist


# ---------- Sistema de Cidades ----------
class Cidade:
    def __init__(self, nome):
        self.nome = nome
        self.bairros = Grafo()

class SistemaCidades:
    def __init__(self):
        self.arvore = ArvoreAVL()

    def cadastrar_cidade(self, id, nome):
        self.arvore.raiz = self.arvore.inserir(self.arvore.raiz, id, Cidade(nome))
        print(f"Cidade {nome} cadastrada! (O(log n))")

    def remover_cidade(self, id):
        self.arvore.raiz = self.arvore.remover(self.arvore.raiz, id)
        print(f"Cidade removida (O(log n))")

    def mostrar_percursos(self):
        print("Pré-ordem:", self.arvore.preorder(self.arvore.raiz))
        print("Em-ordem:", self.arvore.inorder(self.arvore.raiz))
        print("Pós-ordem:", self.arvore.postorder(self.arvore.raiz))

    def adicionar_grafo(self, id, vertices, arestas):
        no = self.arvore.busca(self.arvore.raiz, id)
        if not no:
            print("Cidade não encontrada!")
            return
        for v in vertices:
            no.valor.bairros.add_vertice(v)
        for (u, v, p) in arestas:
            no.valor.bairros.add_aresta(u, v, p)
        print(f"Grafo criado para {no.valor.nome}")

    def explorar_grafo(self, id, inicio):
        no = self.arvore.busca(self.arvore.raiz, id)
        if not no:
            print("Cidade não encontrada!")
            return
        g = no.valor.bairros
        print("BFS:", g.bfs(inicio))
        print("DFS:", g.dfs(inicio))
        print("Dijkstra:", g.dijkstra(inicio))


# ---------- Exemplo ----------
sistema = SistemaCidades()
sistema.cadastrar_cidade(10, "Porto")
sistema.cadastrar_cidade(5, "Rio")
sistema.cadastrar_cidade(15, "Monte")
sistema.mostrar_percursos()

# Criar grafo da cidade "Porto"
bairros = ["Centro", "Norte", "Sul", "Leste", "Oeste"]
arestas = [
    ("Centro", "Norte", 2),
    ("Centro", "Sul", 3),
    ("Centro", "Leste", 1),
    ("Norte", "Leste", 2),
    ("Sul", "Oeste", 4),
]
sistema.adicionar_grafo(10, bairros, arestas)
sistema.explorar_grafo(10, "Centro")


Cidade Porto cadastrada! (O(log n))
Cidade Rio cadastrada! (O(log n))
Cidade Monte cadastrada! (O(log n))
Pré-ordem: [(10, <__main__.Cidade object at 0x7cc68662a180>), (5, <__main__.Cidade object at 0x7cc68662aa50>), (15, <__main__.Cidade object at 0x7cc68662b3e0>)]
Em-ordem: [(5, <__main__.Cidade object at 0x7cc68662aa50>), (10, <__main__.Cidade object at 0x7cc68662a180>), (15, <__main__.Cidade object at 0x7cc68662b3e0>)]
Pós-ordem: [(5, <__main__.Cidade object at 0x7cc68662aa50>), (15, <__main__.Cidade object at 0x7cc68662b3e0>), (10, <__main__.Cidade object at 0x7cc68662a180>)]
Grafo criado para Porto
BFS: ['Centro', 'Norte', 'Sul', 'Leste', 'Oeste']
DFS: ['Centro', 'Leste', 'Norte', 'Sul', 'Oeste']
Dijkstra: {'Centro': 0, 'Norte': 2, 'Sul': 3, 'Leste': 1, 'Oeste': 7}
